In [ ]:
import torchvision
# 1. 准备好的测试集
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter

# 2. 实例化并导入官方自带的 CIFAR-10 测试数据集 [2]
# - 'dataset': 本地存放数据集的相对路径文件夹 [2]
# - train=False: 指定加载 10,000 张测试图片（True 则代表加载 50,000 张训练图） [2]
# - download=True: 若本地没有该数据集，则会自动启动下载 [2]
# - transform=torchvision.transforms.ToTensor(): 一键将原始 PIL 格式图片转为 PyTorch 的浮点 Tensor，
#   同时会自动将像素值从 [0, 255] 归一化对齐到 [0.0, 1.0] 的浮点数区间内 [2]
test_data = torchvision.datasets.CIFAR10(
    'dataset',
    train=False,
    download=True,
    transform=torchvision.transforms.ToTensor()
)

# note 提示：此时可以按住键盘 Ctrl 键并点击 CIFAR10 单词，跳转查看其源码中 __getitem__ 的返回值
# 这样能更深刻地明白为什么下方第 12 行解包时可以同时拿到 img 和 target 两个变量

# 3. 实例化一个自动数据加载器（发牌机） [2]
# - test_data: 被打包的数据源对象 [2]
# - batch_size=64: 每次迭代（发牌）时打包成一个组（Batch），内含 64 张图片 [2]
# - shuffle=True: 在每一轮（Epoch）训练前自动打乱数据顺序，防止模型死记硬背数据顺序 [2]
# - num_workers=0: 0 代表使用单进程（主进程）加载数据，在 Windows 环境下最稳定不易报错 [2]
# - drop_last=True: 10000 张图除以 64 余 16。设为 True 代表直接丢弃最后不规整的 16 张图，
#   确保我们在 for 循环里拿到的每一个 Batch 都雷打不动地是完美的 64 张 [2]
test_loader = DataLoader(
    test_data,
    batch_size=64,
    shuffle=True,
    num_workers=0,
    drop_last=True
)

# ==========================================
# 4. 模拟测试：从数据集（test_data）中读取单个样本
# ==========================================
# 利用方括号 [0] 索引触发 __getitem__ 魔法方法，获取第一个图片样本和它的标签 [2]
img, target = test_data[0]

# 打印第一张图片的形状（Shape）
# 预期输出: torch.Size([3, 32, 32]) ──> 3通道，32x32 分辨率 [2]
print(img.shape)

# 打印第一张图片所对应的分类答案（标签索引，如 3 代表猫）
print(target)

# ==========================================
# 5. 模拟训练循环：从加载器（test_loader）中读取成批样本并写入日志
# ==========================================
# 实例化一个 TensorBoard 记录器，指定本地的日志保存文件夹名为 'dataloader'
writer = SummaryWriter('dataloader')

for epoch in range(2):
  # 初始化一个计数器变量，用于记录当前是“第几步/第几个 Batch”
  step = 0

  # 开启 for 循环，开始迭代并启动自动发牌机
  for data in test_loader:
      # 自动解包当前的这批数据 [2]
      # imgs 形状为 4D 张量: [64, 3, 32, 32] (包含64张3D图片) [2]
      # targets 形状为 1D 向量: [64] (包含64个对应答案) [2]
      imgs, targets = data

      # 提示：你可以去掉下方的 # 注释，来打印观察这批数据的维度流转
      # print(imgs.shape)
      # print(targets)

      # 利用 add_images（注意是复数 s）将这一整包的 64 张图片写入 TensorBoard 日志中
      # - 'test_data': 在浏览器端图像展示的标签名字
      # - imgs: 这包 64 张图片的高能 4D 张量 [2]
      # - step: 步数计数器，在可视化横轴上充当“这是第几批图片”的序号
      writer.add_images(f'Epoch:{epoch}', imgs, step)

      # 步数累加，为下一次发牌做好记录
      step = step + 1

# 训练循环结束后，安全关闭记录器，将缓存中的所有日志刷新并保存到磁盘上，防止数据损坏或丢失
writer.close()

 70%|██████▉   | 119M/170M [10:00<04:02, 213kB/s]

In [ ]:
# 1. 载入 TensorBoard 插件
%load_ext tensorboard

# 2. 关联并启动你刚刚保存日志的 "dataloader" 文件夹
%tensorboard --logdir dataloader